In [1]:
import os
from dotenv import load_dotenv,find_dotenv
_=load_dotenv(find_dotenv())
groq_api_key = os.environ["GROQ_API_KEY"]

In [2]:
from langchain_groq import ChatGroq
llmModel = ChatGroq(
    model="openai/gpt-oss-120b"
)

## Prompts and Prompt Templates

In [3]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} story about {topic}"
)

llmModelPrompt = prompt_template.format(
    adjective = "curious",
    topic="Tesla"
)

response = llmModel.invoke(llmModelPrompt)
print(response.content)

**A Curious Night at Colorado Springs – The Birth of the “Earthquake Machine”**

In the winter of 1899, Nikola Tesla, already famous for his alternating‑current system and his dazzling demonstrations at the 1893 Chicago World’s Fair, set up a modest laboratory on the outskirts of Colorado Springs, Colorado. He had chosen this high‑altitude, thunder‑rich spot because the thin air and the frequent storms made it an ideal place to test his most daring ideas about wireless power and the hidden forces of the Earth.

### The Setting

Tesla’s laboratory was a wooden shack perched on a hill overlooking the town. Inside, a forest of copper coils, brass terminals, and humming transformers filled the space. The most prominent piece of equipment was a massive **magnifying transmitter**—a 25‑foot‑tall tower of copper and steel that could generate voltages of up to a million volts and frequencies in the kilohertz range. By night, the tower would glow with a ghostly blue corona, crackling like a stor

In [4]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system","You are an {profession} expert on {topic}."),
        ("human","Hello, Mr. {profession}, can you please answer a question?"),
        ("ai","Sure!"),
        ("human","{user_input}"),
    ]
)

messages = chat_template.format(
    profession="Historian",
    topic="The Kennedy family",
    user_input="How many grandchildren had Joseph P. Kennedy?"
)

response = llmModel.invoke(messages)
print(response.content)

**Joseph P. Kennedy Sr. (1888‑1969) was the patriarch of a family that eventually produced a dozen‑plus grandchildren.**  

Below is a concise breakdown of the children of each of his nine offspring and the total number of grandchildren (including those who died in infancy).

| Kennedy sibling | Grandchildren (names) | Count |
|-----------------|-----------------------|-------|
| **John F. Kennedy** (1917‑1963) | Caroline Kennedy (b. 1957) – John F. Kennedy Jr. (1960‑1999) | **2** |
| **Rosemary Kennedy** (1918‑2001) | – (no children) | **0** |
| **Kathleen Kennedy** (1920‑1948) | – (no children) | **0** |
| **Eunice Kennedy Shriver** (1921‑2000) | Maria Shriver, Robert “Bob” Shriver, Timothy Shriver, Anthony Shriver, Christopher Shriver | **5** |
| **Patricia Kennedy Lawford** (1924‑2006) | Christopher Lawford (1955‑2018), Sydney Lawford (b. 1956), *Patricia Lawford* (daughter who died shortly after birth) | **3** |
| **Jean Kennedy Smith** (1928‑2020) | – (no children) | **0** |
| **

# Old Way

In [5]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content=(
                "You are an Historian expert on the Kennedy family."
            )
        ),
        HumanMessagePromptTemplate.from_template("{user_input}"),
    ]
)

messages = chat_template.format_messages(
    user_input = "Name the children and grandchildren of Joseph P. Kennedy?"
)

response = llmModel.invoke(messages)
print(response.content)

# Few Shot Prompting

In [6]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples=[
    {"input":"hi!","output":"hola!"},
    {"input":"bye!","output":"adios!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human","{input}"),
        ("ai","{output}")
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are an English-Spanish translator."),
        few_shot_prompt,
        ("human","{input}"),
    ]
)

#### Chains

In [7]:
chain = final_prompt | llmModel 

response = chain.invoke({"input":"My name is Abhishekh. I am 24 years old."})
print(response.content)

Me llamo Abhishekh. Tengo 24 años.


## Output Parsers

In [8]:
from langchain_groq import ChatGroq

chatmodel = ChatGroq(model="openai/gpt-oss-120b")

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.json import SimpleJsonOutputParser

json_prompt = PromptTemplate.from_template(
    "Return a JSON object with an `answer` key that answers the following question: {question}"
)

json_parser = SimpleJsonOutputParser()

json_chain = json_prompt | llmModel | json_parser

In [17]:
res = json_chain.invoke({"question":"What is the smallest country?"})

In [18]:
res

{'answer': 'Vatican City'}

In [19]:
type(res)

dict

### Optionally, you can use Pydantic to define a custom output format

In [20]:
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.json import SimpleJsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

In [24]:
# Define a Pydantic Object with the desired output format.

class Joke(BaseModel):
    setup:str = Field(description="question to set up a joke")
    punchline:str = Field(description="answer to resolve a joke")

In [25]:
# Define the parser referring the Pydantic Object
parser = SimpleJsonOutputParser(pydantic_object=Joke)

# Add the parser format instructions in the prompt definition.
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables = ["query"],
    partial_variables={"format_instructions":parser.get_format_instructions()},
)

# Create a chain with the prompt and the parser
chain = prompt | chatmodel | parser

chain.invoke({"query":"Tell me a joke on narendra modi."})


{'setup': 'Why did Narendra Modi bring a ladder to the Parliament?',
 'punchline': 'Because he heard the debates were reaching new heights!'}

In [31]:
from groq import Groq

client = Groq()

chat_completion = client.chat.completions.create(
    #
    # Required parameters
    #
    messages=[
        # Set an optional system message. This sets the behavior of the
        # assistant and can be used to provide specific instructions for
        # how it should behave throughout the conversation.
        {"role": "system", "content": "you are a helpful assistant."},
        # Set a user message for the assistant to respond to.
        {
            "role": "user",
            "content": "Explain the importance of low latency LLMs",
        },
    ],
    # The language model which will generate the completion.
    model="openai/gpt-oss-120b",
    #
    # Optional parameters
    #
    # Controls randomness: lowering results in less random completions.
    # As the temperature approaches zero, the model will become deterministic
    # and repetitive.
    temperature=0.5,
    # The maximum number of tokens to generate. Requests can use up to
    # 2048 tokens shared between prompt and completion.
    max_tokens=1024,
    # Controls diversity via nucleus sampling: 0.5 means half of all
    # likelihood-weighted options are considered.
    top_p=1,
    # A stop sequence is a predefined or user-specified text string that
    # signals an AI to stop generating content, ensuring its responses
    # remain focused and concise. Examples include punctuation marks and
    # markers like "[end]".
    stop=None,
    # If set, partial message deltas will be sent.
    stream=False,
)

# Print the completion returned by the LLM.
print(chat_completion.choices[0].message.content)

### Why Low‑Latency Large Language Models (LLMs) Matter

| **Aspect** | **What “low latency” means** | **Why it’s important** |
|------------|-----------------------------|------------------------|
| **User Experience** | The time between a user’s input and the model’s response is measured in milliseconds to a few seconds. | • Faster replies feel more natural and conversational.<br>• Reduces frustration in chat‑bots, virtual assistants, and search interfaces.<br>• Enables “instant‑answer” experiences that compete with traditional UI patterns (e.g., autocomplete, instant search). |
| **Real‑Time Applications** | Systems that must react within strict time budgets (e.g., < 100 ms for voice assistants, < 200 ms for gaming NPCs). | • Voice‑controlled devices need quick turn‑around to avoid awkward pauses.<br>• Autonomous vehicles, robotics, and IoT devices rely on immediate language understanding for safety‑critical decisions.<br>• Live transcription, captioning, and translation must keep u

In [29]:
# Print the completion returned by the LLM.
print(chat_completion.choices[0].message.content)

In [34]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json

# Initialize Groq LLM
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.7
)

# Define the expected JSON structure
class ProductDetails(BaseModel):
    name: str
    price: float
    features: list[str]

parser = JsonOutputParser(pydantic_object=ProductDetails)

# Create a simple prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract product details into JSON with this structure:
        {{
            "name": "product name here",
            "price": number_here_without_currency_symbol,
            "features": ["feature1", "feature2", "feature3"]
        }}"""),
    ("user", "{input}")
])

# Create the chain that guarantees JSON output
chain = prompt | llm | parser

def parse_product(description: str) -> dict:
    result = chain.invoke({"input": description})
    print(json.dumps(result, indent=2))

        
# Example usage
description = """The Kees Van Der Westen Speedster is a high-end, single-group espresso machine known for its precision, performance, 
and industrial design. Handcrafted in the Netherlands, it features dual boilers for brewing and steaming, PID temperature control for 
consistency, and a unique pre-infusion system to enhance flavor extraction. Designed for enthusiasts and professionals, it offers 
customizable aesthetics, exceptional thermal stability, and intuitive operation via a lever system. The pricing is approximatelyt $14,499 
depending on the retailer and customization options."""

parse_product(description)



{
  "name": "Kees Van Der Westen Speedster",
  "price": 14499,
  "features": [
    "dual boilers for brewing and steaming",
    "PID temperature control for consistency",
    "unique pre-infusion system to enhance flavor extraction",
    "handcrafted in the Netherlands",
    "customizable aesthetics",
    "exceptional thermal stability",
    "intuitive operation via a lever system"
  ]
}


In [37]:
from groq import Groq

client = Groq()

stream = client.chat.completions.create(
    #
    # Required parameters
    #
    messages=[
        # Set an optional system message. This sets the behavior of the
        # assistant and can be used to provide specific instructions for
        # how it should behave throughout the conversation.
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        # Set a user message for the assistant to respond to.
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],

    # The language model which will generate the completion.
    model="qwen/qwen3.8-27b",

    #
    # Optional parameters
    #

    # Controls randomness: lowering results in less random completions.
    # As the temperature approaches zero, the model will become deterministic
    # and repetitive.
    temperature=0.5,

    # The maximum number of tokens to generate. Requests can use up to
    # 2048 tokens shared between prompt and completion.
    max_completion_tokens=1024,

    # Controls diversity via nucleus sampling: 0.5 means half of all
    # likelihood-weighted options are considered.
    top_p=1,

    # A stop sequence is a predefined or user-specified text string that
    # signals an AI to stop generating content, ensuring its responses
    # remain focused and concise. Examples include punctuation marks and
    # markers like "[end]".
    stop=None,

    # If set, partial message deltas will be sent.
    stream=True,
)

# Print the incremental deltas returned by the LLM.
for chunk in stream:
    print(chunk.choices[0].delta.content, end="")

The importance of fast language models (LLMs) cannot be overstated, as speed is often the critical factor that determines whether an AI system is usable, scalable, and economically viable. While raw intelligence (accuracy, reasoning, creativity) is the foundation, **latency and throughput** are the enablers that bring that intelligence to the masses.

Here’s a breakdown of why speed matters in several key dimensions:

### 1. User Experience (UX) and Interactivity
*   **Perceived Responsiveness:** Human conversation is fast. If a model takes 5–10 seconds to respond, the interaction feels clunky, robotic, and frustrating. Fast models (generating tokens in real-time) create a seamless, conversational flow that feels natural.
*   **Iterative Refinement:** Users often refine prompts based on initial outputs. Faster responses allow for quicker "prompt engineering" loops, enabling users to iterate, correct, and improve results in real-time.
*   **Real-Time Applications:** Use cases like live 